# MAP-SAAS と加法 GP モデル

この Notebook では、高次元問題向けの3つの exact GP 系モデル `AdditiveMapSaasSingleTaskGP`、`EnsembleMapSaasSingleTaskGP`、`OrthogonalAdditiveGP` を同じ合成データで比較します。

In [ ]:
import torch
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import (
    AdditiveMapSaasSingleTaskGP,
    EnsembleMapSaasSingleTaskGP,
    OrthogonalAdditiveGP,
)

torch.manual_seed(0)
dtype = torch.double

## 1. 高次元の合成関数

10次元の入力を用意しますが、目的関数に直接効くのは一部の次元だけにしています。高次元かつ疎な関連性を持つ状況を簡略化した例です。

In [ ]:
n, d = 36, 10
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (
        torch.sin(2 * torch.pi * X[..., 0])
        + 0.7 * (X[..., 3] - 0.5) ** 2
        - 0.5 * X[..., 6]
    ).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

## 2. Additive MAP-SAAS

SAAS の疎な関連性の考え方を MAP 推定で扱う加法モデルです。Fully Bayesian SAAS の NUTS より軽量な選択肢として利用できます。

In [ ]:
additive_saas = AdditiveMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(additive_saas.raw_train_X.shape, additive_saas.raw_train_Y.shape)
print('supports_mll =', additive_saas.supports_mll)
fit_gpytorch_mll(additive_saas.make_mll())

## 3. Ensemble MAP-SAAS

複数の MAP-SAAS 設定をアンサンブルとして扱うモデルです。Fully Bayesian 推論より計算を抑えつつ、単一の MAP 推定への依存を緩和したい場合に候補になります。

In [ ]:
ensemble_saas = EnsembleMapSaasSingleTaskGP(train_X, train_Y, num_taus=3)
print(ensemble_saas.raw_train_X.shape, ensemble_saas.raw_train_Y.shape)
fit_gpytorch_mll(ensemble_saas.make_mll())

## 4. Orthogonal Additive GP

目的関数に一次または低次の加法構造があるという仮定を明示的に利用するモデルです。ここでは `second_order=False` とし、一次の加法構造を使います。

In [ ]:
orthogonal = OrthogonalAdditiveGP(train_X, train_Y, second_order=False)
print(orthogonal.raw_train_X.shape, orthogonal.raw_train_Y.shape)
fit_gpytorch_mll(orthogonal.make_mll())

## 5. 同じ断面で posterior mean を比較

他の入力を 0.5 に固定し、`x0` だけを変化させた断面で各モデルの posterior mean を確認します。

In [ ]:
grid = torch.linspace(0, 1, 80, dtype=dtype)
X_test = torch.full((80, d), 0.5, dtype=dtype)
X_test[:, 0] = grid
models = {
    'additive MAP-SAAS': additive_saas,
    'ensemble MAP-SAAS': ensemble_saas,
    'orthogonal additive': orthogonal,
}
for name, model in models.items():
    with torch.no_grad():
        mean = model.posterior(X_test).mean.squeeze(-1)
    print(name, 'mean range =', (float(mean.min()), float(mean.max())))

## 6. モデル選択の目安

- **Additive MAP-SAAS**: 高次元で疎な関連性と加法構造の両方が妥当と考えられる場合に向きます。
- **Ensemble MAP-SAAS**: 複数の MAP-SAAS 設定をまとめ、full NUTS より実用的な計算量にしたい場合の候補です。
- **OrthogonalAdditiveGP**: 一次または低次の加法分解そのものが妥当なモデリング仮定である場合に向きます。
- Fully Bayesian SAAS と異なり、これらは exact GP の `make_mll()` による学習フローを利用できます。